# Finno-Ugric Dataset: Category Visualizations

This notebook loads the CSV files in this folder and visualizes the most frequent categories in each dataset.

In [15]:
from pathlib import Path
import pandas as pd
from IPython.display import HTML, display

try:
    import plotly.express as px
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly'])
    import plotly.express as px

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

In [16]:
data_dir = Path('.')

files = {
    'places': 'finno-ugric-object-places.csv',
    'materials': 'finno-ugric-object-materials.csv',
    'collectors': 'finno-ugric-object-collectors.csv',
    'relations': 'finno_ugric_object_relations.csv',
    'objects_sheet': 'finno-ugric-objects-dataset.xlsx - Sheet1.csv'
}

dfs = {}
for name, filename in files.items():
    path = data_dir / filename
    if path.exists():
        dfs[name] = pd.read_csv(path)

print('Loaded tables:')
for name, df in dfs.items():
    print(f'- {name}: {df.shape[0]:,} rows x {df.shape[1]} columns')

Loaded tables:
- places: 14,621 rows x 13 columns
- materials: 15,178 rows x 3 columns
- collectors: 16,238 rows x 4 columns
- relations: 889 rows x 9 columns
- objects_sheet: 10,079 rows x 28 columns


In [17]:
summary = []
for name, df in dfs.items():
    summary.append({
        'table': name,
        'rows': df.shape[0],
        'columns': df.shape[1],
        'categorical_columns': ', '.join(df.select_dtypes(include=['object']).columns.tolist())
    })

pd.DataFrame(summary).sort_values('table').reset_index(drop=True)

,table,rows,columns,categorical_columns
0,collectors,16238,4,"collector, collector_normalized"
1,materials,15178,3,"material, material_category"
2,objects_sheet,10079,28,"museal_number, image_url, object_url, title, e..."
3,places,14621,13,"place_norm, country, region, district, Distric..."
4,relations,889,9,relation_id


## Category Plot Helper

The function below draws interactive top-N category charts.
Hover over any bar to see the exact count.
A table of the same graph values is shown before each chart.

In [21]:
def plot_top_categories(df, table_name, columns, top_n=15):
    valid_cols = [c for c in columns if c in df.columns]
    if not valid_cols:
        print(f'No requested categorical columns found in: {table_name}')
        return

    for col in valid_cols:
        counts = (
            df[col]
            .astype('string')
            .str.strip()
            .replace({'': pd.NA})
            .dropna()
            .value_counts()
            .head(top_n)
        )

        if counts.empty:
            print(f'No non-null values in {table_name}.{col}')
            continue

        table_df = counts.reset_index()
        table_df.columns = ['category', 'count']

        print(f"Top {min(top_n, len(table_df))} values for {table_name}.{col}")
        display(table_df)

        plot_df = table_df.sort_values('count', ascending=True)

        fig = px.bar(
            plot_df,
            x='count',
            y='category',
            orientation='h',
            title=f"{table_name} - Top {min(top_n, len(plot_df))} values in {col}",
            labels={'count': 'Count', 'category': col}
        )

        fig.update_traces(
            hovertemplate=f"{col}: %{{y}}<br>Count: %{{x}}<extra></extra>"
        )
        fig.update_layout(height=max(420, 28 * len(plot_df) + 180))

        # Render as HTML to avoid renderer dependency issues in some kernels.
        display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))

In [22]:
recommended_category_columns = {
    'places': ['country', 'region', 'district', 'local_admin', 'settlement', 'coords_source'],
    'materials': ['material_category', 'material'],
    'collectors': ['collector_normalized', 'collector'],
    'relations': ['relation_id'],
    'objects_sheet': []
}

for table_name, df in dfs.items():
    cols = recommended_category_columns.get(table_name, [])

    # Fallback: use all object columns except likely ID fields if no explicit list is provided.
    if not cols:
        cols = [
            c for c in df.select_dtypes(include=['object']).columns
            if 'id' not in c.lower()
        ][:6]

    print(f'\n=== {table_name} ===')
    plot_top_categories(df, table_name, cols, top_n=15)


=== places ===
Top 10 values for places.country


,category,count
0,Venemaa,12805
1,Läti,677
2,Ukraina,234
3,Soome,213
4,Ungari,70
5,NSV,51
6,Rootsi,41
7,Moldova,1
8,Rumeenia,1
9,Komi NSV,1


Top 15 values for places.region


,category,count
0,Leningradi oblast,2396
1,Mari ANSV,1702
2,Udmurdi ANSV,1679
3,Mordva ANSV,978
4,Tjumeni oblast,922
5,Kirovi oblast,770
6,Tatari ANSV,532
7,Karjala ANSV,461
8,Komi ANSV,443
9,Vologda oblast,352


Top 15 values for places.district


...

,category,count
0,peapael,311
1,põll,257
2,käterätt,184
3,"särk, naiste",172
4,nõu,168
5,särk,161
6,kleit,160
7,pealinik,153
8,korv,147
9,vöö,140


Top 15 values for objects_sheet.ethnic_group_name


,category,count
0,udmurdi,1661
1,mari,1519
2,vepsa,1074
3,isuri,966
4,handi,759
5,karjala,653
6,mordva,580
7,komi,568
8,liivi,549
9,ersa,420


Top 15 values for objects_sheet.collection_year


,category,count
0,1987,649
1,1989,626
2,1988,585
3,1983,563
4,1986,540
5,1975,539
6,1977,519
7,1990,366
8,1982,358
9,1937,354


## Optional: One Specific Category Check

Use this cell to drill down into a specific column manually.

In [20]:
# Example:
# dfs['places']['country'].value_counts(dropna=True).head(20)
# dfs['materials']['material_category'].value_counts(dropna=True).head(20)